---
execute:
  skip: true
---
# 📈 Getting Constant Maturity Yields From FRED
<br>

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Four-Chapter-One/blob/master/colab/Colab_Getting_Constant_Maturity_Yields_From_FRED.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Four-Chapter-One/master?urlpath=lab/tree/notebooks/Getting_Constant_Maturity_Yields_From_FRED.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Four-Chapter-One/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>
</div>

This notebook puts our secure key management and data pipeline to FRED to work. Before we dive in, you need to have some familiarity with FRED series IDs and how to extract them. As you will see, the `get_series` method of `FredReader` deftly handles incorrect series IDs and can be used to sort out the correct ones.

Start off by watching this short video on the FRED database and its series IDs


[![Getting Series IDs From Fred](https://img.youtube.com/vi/ZP1hKFRZAz4/0.jpg)](https://youtu.be/ZP1hKFRZAz4)

:::{important} [ ▼ ] How to use this page: Run, Copy, & Download
:class: dropdown

<ul>
  <li><b>⏻ Run code right here:</b> Click the <b>Power Button</b> icon at the top of the screen to activate <b>Live Code</b>.</li>
  <li><b>📋 Copy code:</b> Hover over any code block and click the <b>Clipboard icon</b> in the top-right corner.</li>
  <li><b>📥 Download this file:</b> Click the <b>Download icon</b> (downward arrow) at the top right of the screen to save this exact notebook to your computer.</li>
</ul>
:::

:::{important} 🤔 Notebook Setup: Why the "Try/Except" Imports?
:class: dropdown

**The Goal:**
To ensure this notebook runs perfectly whether you are using **Google Colab**, a local **Jupyter instance**, or a remote server without you having to manually install software.

* **External Libraries:** NumPy and Pandas are the "heavy hitters" for data. They aren't always installed by default.
* **The `try/except` Logic:** This is a safety net.
    1. We **try** to import the library.
    2. If it fails (because it's not installed), the **except** block triggers a `%pip install` to download it automatically.
* **Aliasing (`as np`):** We rename `numpy` to `np` to save keystrokes. In professional finance code, `np` and `pd` are the universal shorthand.
  
:::

## 🛠️ Preparing the Notebook

<details>
<summary><b>👉 Click to Expand: 📦 Importing Libraries, Modules, and Functions</b></summary>

As a best practice, we always begin by importing our necessary dependencies in the very first code cell. Notice how lightweight our imports are here: just the `pandas` library. ✨

This simplicity is a direct result of using the `financial_quant` package, which handles the heavy lifting behind the scenes. 🏗️ By keeping complicated setup details out of sight, we ensure the spotlight remains focused exactly where it belongs—on the core analysis. 🎯

```python
try:
    import pandas as pd
except:
    %pip -q install pandas
    import pandas as pd
```
**👀 Keep an eye out**: As we progress, pay attention to how the `financial_quant` package is imported as `fq`, and how every reference to its functions begins with fq.. 💡 This follows the exact same standard practice we demonstrated in Chapter One with NumPy (np) and Pandas (pd).

</details>


In [ ]:
try:
    import pandas as pd
except:
    %pip -q install pandas
    import pandas as pd

## 📦 Getting Functions from financial_quant package

:::{important} 🔌 Dynamic Verification: How Remote Version Checking Works
:class: dropdown

**The Logic:**
Instead of blindly reinstalling packages or guessing if our local tools are up to date, we are dynamically reaching out to GitHub to inspect the live code. By scanning the remote file on the fly, the notebook can verify exactly which version of the quantitative toolkit is currently published before we use it.

**The Workflow:**
1. **Request:** The code acts like an automated web browser. It uses Python's `urllib` to visit the GitHub URL and request the raw text of the Python script.
2. **Decode:** The internet sends data back as raw bytes (1s and 0s). We use `.decode('utf-8')` to translate that raw data into a readable Python string.
3. **Search & Extract:** We use a Regular Expression (`regex`) to scan the document, pinpointing the exact line where the version is declared and extracting the number.
4. **Verify:** The notebook compares this remote version against your local installation to ensure you have the latest financial models. 
5. **Import & Route:** Once verified, we `import financial_quant as fq`. Behind the scenes, an `__init__.py` file routes the complex tools from deeply nested folders up to the surface.
6. **Execute:** You can now confidently type `fq.one_y_axis()` or `fq.calc_ytm()`, knowing you are running the most up-to-date code.

**Why do this?**
It makes your notebooks highly intelligent. Rather than relying on static installations, the notebook becomes self-aware—it can check if its math models are perfectly synced with the master branch and alert you if a critical update is missing. It is a lightweight way to guarantee reproducibility without cluttering the environment.
:::

In [ ]:
# Use Python's built-in urllib to fetch the remote setup script
import urllib.request

# Download, decode, and execute the updater script directly from GitHub
updater_url = "https://raw.githubusercontent.com/PatrickJHess/quant_repo/master/fq_updater.py"
exec(urllib.request.urlopen(updater_url).read().decode('utf-8'))

# Run the installer/updater function and alias the package as 'fq'
fq = import_financial_quant()

## 🏦 🆔 Series IDs for constant maturity yields and the secured overnight funding rate SOFR

Generate a potential list of Series IDs for constant maturity yields as `f` strings..

*   **Monthly Series**: all monthly maturities between one and eleven months

```
monthly_ids = [f"DGS{i}MO" for i in range(1, 12)]
```


*   **Yearly Series**: all years between one and thirty years

```
yearly_ids = [f"DGS{i}" for i in range(1, 31)]
```

Secured overnight Id is 'SOFR'.  The list of all IDs is series_id.

In [ ]:
# Generate months 1-11 and years 1-30 programmatically
monthly_ids = [f"DGS{i}MO" for i in range(1, 12)]
yearly_ids = [f"DGS{i}" for i in range(1, 31)]

# Combine everything together with SOFR
series_ids = ['sofr'] + monthly_ids + yearly_ids

## 🔗 Accessing constant maturity yields for May 2026 with `FredReader`.



*   **Create an instance of the FredRreder**
      ```
      fred_data=FredReader()
      ```
*   **FredReader method `get_series`**
    * **`series_id`** is required and be a string or an iterable of strings.
    * **`startng_date` and `end_date`** are optional.
    * **`ttl`** time for cache to live defaults to 7 days.   
      ```
      yield_data=fred_data.get_series(series_ids,start_date='2026-05-01')
      ```



In [ ]:
# create an instance of FredReader
fred_data=fq.FredReader()

# call the method for the class
yield_data=fred_data.get_series(series_ids,start_date='2026-05-01')

In [ ]:
display(yield_data)



:::::{admonition} ✍️ FRED Data Challenge


*   Access Overnight Secured Funding Rate And Constant Maturity Yields (Par Yields) For Thirty Year Maturity between January 1, 2026 and May 20, 2026.
*   Did you access series from cache or FRED?

:::{tip}
Use the first and last column heads of `yield_data` as series IDs.
:::
:::{dropdown} ✅ Example Of Solution
## Example of Code


```py
 # assuming imports of this notebook 
 fred_data=fq.FredReader()
 series_ids=[yield_data.columns[0],yield_data.columns[-1]]
 fred_data.get_series(series_ids,start_date='2026-01-01',end_date='2026-05-20')
```
:::

:::{dropdown} 🎯 Answer
The previous request for FRED data specified May 1, 2026 as the start date. If that was your only request or your last request was seven days ago,the cache for the two series `sofr` and `DGS30` is updated with an API request to FRED.

:::::

